In [1]:
!apt-get install -y git
!pip install uv

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 61.4 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/Flora-jia-jfr/CPS572_Sp26_mini_project.git /content/project
%cd /content/project

Cloning into '/content/project'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 26 (delta 0), reused 0 (delta 0), pack-reused 22 (from 2)
Receiving objects: 100% (26/26), 21.01 KiB | 7.00 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/project


In [3]:
!uv pip install -r requirements.txt --system

Using Python 3.12.13 environment at: /usr
Resolved 142 packages in 9.69s
Prepared 77 packages in 59.60s
Uninstalled 32 packages in 1.73s
Installed 77 packages in 1.70s
 + aioboto3==15.5.0
 + aiobotocore==2.25.1
 - aiofiles==24.1.0
 + aiofiles==25.1.0
 - aiohttp==3.13.5
 + aiohttp==3.13.4
 + aioitertools==0.13.0
 + backoff==2.2.1
 - beautifulsoup4==4.13.5
 + beautifulsoup4==4.14.3
 + boto3==1.40.61
 + botocore==1.40.61
 - charset-normalizer==3.4.7
 + charset-normalizer==3.4.6
 + chz==0.4.0
 - click==8.3.2
 + click==8.2.1
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.0
 + cuda-toolkit==13.0.2
 - datasets==4.0.0
 + datasets==4.8.4
 - debugpy==1.8.15
 + debugpy==1.8.20
 - dill==0.3.8
 + dill==0.4.1
 - fsspec==2025.3.0
 + fsspec==2025.9.0
 - hf-xet==1.4.3
 + hf-xet==1.4.2
 - huggingface-hub==1.10.1
 + huggingface-hub==1.8.0
 + ijson==3.5.0
 + inspect-ai==0.3.170
 + inspect-evals==0.3.106
 + instruction-following-eval==0.1.0 (from git+https://github.com/josejg/instruction_following_eval@0

In [ ]:
import os
os.environ["TINKER_API_KEY"] = ""
os.environ["PYTHONPATH"] = "/content/project"

In [ ]:
# !python evaluation/eval_all.py --base_models meta-llama/Llama-3.2-3B --limit 5
# !python evaluation/eval_all.py --base_models meta-llama/Llama-3.2-3B
# !python evaluation/eval_all.py --base_models meta-llama/Llama-3.2-1B meta-llama/Llama-3.2-3B meta-llama/Llama-3.1-8B


############################################################
# BASELINE (core): meta-llama/Llama-3.2-3B
############################################################
INFO:__main__:Evaluating: base:meta-llama/Llama-3.2-3B
INFO:__main__:Renderer: role_colon | temperature=0.0 top_p=1.0
INFO:__main__:============================================================
INFO:__main__:TASK 1/3: IFEval (Instruction Following)
INFO:__main__:============================================================
INFO:evaluation.eval_ifeval:Model: meta-llama/Llama-3.2-3B | Renderer: role_colon
INFO:tinker.lib.public_interfaces.service_client:ServiceClient initialized for session 337a080f-fe7d-5d34-a6b0-d1feb0eba566
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
tokenizer_config.json: 55.4kB [00:00, 9.07MB/s]
tokenizer.json: 9.09MB [00:00, 83.8MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.04MB/s]
INFO:datasets:TensorFlow version 2.19.0 available.
INFO:datasets:JAX version 0.7.2 available.
Loading 

In [ ]:
# # ──────────────────────────────────────────────
# # 数据加载函数
# # ──────────────────────────────────────────────

# def load_gsm8k_data(num_samples=7473, upsample_factor=2):
#     """
#     加载 GSM8K 数学题数据（全量 + 上采样）。
#     原始 answer 字段自带 CoT + #### 格式，直接使用。
#     上采样是为了补偿 GSM8K 数据量远少于其他两类的问题。
#     """
#     print("Loading GSM8K...")
#     dataset = load_dataset("openai/gsm8k", "main", split="train")
#     n = min(num_samples, len(dataset))
#     dataset = dataset.select(range(n))

#     # 打印前2条确认格式
#     print("  [格式验证] GSM8K前2条answer样例：")
#     for i in range(min(2, len(dataset))):
#         print(f"    [{i}] {dataset[i]['answer'][:120]}...")

#     conversations = []
#     for item in dataset:
#         conversations.append([
#             {
#                 "role": "system",
#                 "content": (
#                     "You are a math problem solver. "
#                     "Think step by step and show your reasoning. "
#                     "At the end, write your final answer in the format: #### <number>"
#                 )
#             },
#             {"role": "user", "content": item["question"]},
#             {"role": "assistant", "content": item["answer"]},
#         ])

#     # 上采样：复制整个列表 upsample_factor 次
#     conversations = conversations * upsample_factor
#     print(f"  Loaded {n} GSM8K examples × {upsample_factor} = {len(conversations)} total")
#     return conversations


# def load_ifeval_data(num_samples=5000):
#     """
#     加载 Tulu-3 指令跟随数据。
#     Tulu-3 包含大量带可验证约束的指令，与 IFEval 风格最接近。
#     数据集很大（~939K），只采样一个子集。
#     """
#     print("Loading IFEval data (Tulu-3)...")
#     try:
#         dataset = load_dataset("allenai/tulu-3-sft-mixture", split="train")

#         # 随机采样
#         indices = np.random.choice(len(dataset), size=min(num_samples, len(dataset)), replace=False)
#         dataset = dataset.select(indices.tolist())

#         conversations = []
#         skipped = 0
#         for item in dataset:
#             messages = item.get("messages", [])
#             # Tulu-3 已经是多轮对话格式，直接使用
#             if len(messages) >= 2:
#                 conversations.append(messages)
#             else:
#                 skipped += 1

#         print(f"  Loaded {len(conversations)} Tulu-3 examples (skipped {skipped} malformed)")
#         return conversations

#     except Exception as e:
#         print(f"  [WARNING] Failed to load Tulu-3: {e}")
#         print("  Falling back to Alpaca (suboptimal for IFEval)...")
#         return _load_alpaca_fallback(num_samples)


# def _load_alpaca_fallback(num_samples=2000):
#     """Tulu-3 加载失败时的备用数据集"""
#     dataset = load_dataset("tatsu-lab/alpaca", split="train")
#     conversations = []
#     for item in dataset.select(range(min(num_samples, len(dataset)))):
#         user_content = item["instruction"]
#         if item["input"].strip():
#             user_content += f"\n\n{item['input']}"
#         conversations.append([
#             {"role": "user", "content": user_content},
#             {"role": "assistant", "content": item["output"]},
#         ])
#     print(f"  Loaded {len(conversations)} Alpaca examples (fallback)")
#     return conversations


# def load_code_data(num_samples=5000):
#     print("Loading code data (OpenCodeInstruct)...")
#     try:
#         # 不用 streaming，直接指定只加载第一个分片
#         dataset = load_dataset("sahil2801/CodeAlpaca-20k", split="train")


#         conversations = []
#         skipped = 0
#         for item in dataset:
#             instruction = item.get("input", "").strip()
#             response = item.get("output", "").strip()

#             if not instruction or not response:
#                 skipped += 1
#                 continue

#             # 去掉 markdown 包裹
#             if response.startswith("```python"):
#                 response = response[len("```python"):].strip()
#             elif response.startswith("```"):
#                 response = response[3:].strip()
#             if response.endswith("```"):
#                 response = response[:-3].strip()

#             conversations.append([
#                 {
#                     "role": "system",
#                     "content": (
#                         "You are an expert Python programmer. "
#                         "Write clean, correct Python code. "
#                         "Output only the raw Python code, do NOT use markdown code blocks."
#                     )
#                 },
#                 {
#                     "role": "user",
#                     "content": instruction + "\n\nOnly output the raw Python code, do NOT use markdown code blocks or any explanation."
#                 },
#                 {
#                     "role": "assistant",
#                     "content": response
#                 },
#             ])

#         print(f"  Loaded {len(conversations)} code examples (skipped {skipped} malformed)")
#         return conversations

#     except Exception as e:
#         print(f"  [WARNING] Failed to load OpenCodeInstruct: {e}")
#         print("  Falling back to MBPP...")
#         return _load_mbpp_fallback(num_samples)


# def _load_mbpp_fallback(num_samples=1000):
#     dataset = load_dataset("google-research-datasets/mbpp", "full", split="train")
#     conversations = []
#     for item in dataset.select(range(min(num_samples, len(dataset)))):
#         prompt = (
#             f"Write a Python function to solve the following problem:\n{item['text']}\n\n"
#             f"Your function should pass these tests:\n"
#             + "\n".join(item["test_list"][:2])
#             + "\n\nOnly output the raw Python code, do NOT use markdown code blocks."
#         )
#         conversations.append([
#             {"role": "user", "content": prompt},
#             {"role": "assistant", "content": item["code"]},  # MBPP 的 code 字段本身就是纯代码
#         ])
#     print(f"  Loaded {len(conversations)} MBPP examples (fallback)")
#     return conversations

Overwriting /content/project/evaluation/train_and_publish.py


In [ ]:
# # load data from preprocessed dataset
# print("Loading pre-processed training data...")
# with open("training_data.jsonl") as f:
#     training_data = [json.loads(line) for line in f]
# print(f"  Loaded {len(training_data)} examples")

Loading pre-processed training data...
  Loaded 30000 examples


In [ ]:
# # Verify task distribution
# from collections import Counter
# task_names = {0: "GSM8K", 1: "IFEval", 2: "Code"}
# dist = Counter(ex["task_id"] for ex in training_data)
# for tid, name in task_names.items():
#     print(f"  {name}: {dist.get(tid, 0)}")

# # Build labeled list directly from task_id field
# labeled = [(ex["messages"], ex["task_id"]) for ex in training_data]

  GSM8K: 10000
  IFEval: 10000
  Code: 10000


In [40]:
%%writefile evaluation/train_and_publish.py
"""
Train a model with real data from GSM8K, Tulu-3, and OpenCodeInstruct.
Multi-task fine-tuning for IFEval, GSM8K, and HumanEval benchmarks.
"""

import argparse
import json
import os
import math

import numpy as np
import tinker
from tinker import types
from tinker_cookbook import model_info, renderers
from tinker_cookbook.supervised.data import conversation_to_datum
from tinker_cookbook.tokenizer_utils import get_tokenizer
from datasets import load_dataset

# MODEL = "meta-llama/Llama-3.2-3B"
MODEL = "meta-llama/Llama-3.1-8B"

EVAL_DIR = os.path.dirname(os.path.abspath(__file__))

# ──────────────────────────────────────────────
# 学习率 Warmup 工具
# ──────────────────────────────────────────────

def get_lr_with_warmup(step, base_lr, warmup_steps, total_steps, min_lr_ratio=0.1):
    """
    线性 warmup + cosine decay 学习率调度。
    - 前 warmup_steps 步：从 0 线性升到 base_lr
    - 之后：cosine 衰减到 base_lr * min_lr_ratio
    """
    if step < warmup_steps:
        return base_lr * (step + 1) / warmup_steps
    else:
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        cosine_factor = 0.5 * (1 + math.cos(math.pi * progress))
        return base_lr * (min_lr_ratio + (1 - min_lr_ratio) * cosine_factor)


# ──────────────────────────────────────────────
# 数据遍历工具
# ──────────────────────────────────────────────

class ShuffledDataLoader:
    """
    每个 epoch 结束后自动 reshuffle，避免边界重复采样。
    同时按 task_id 记录每个样本的任务类型，方便分任务统计 loss。
    """
    def __init__(self, data_with_task_ids, batch_size, seed=42):
        self.data = data_with_task_ids   # list of (datum, task_id)
        self.batch_size = batch_size
        self.rng = np.random.default_rng(seed)
        self.indices = np.arange(len(self.data))
        self.rng.shuffle(self.indices)
        self.pos = 0
        self.epoch = 0

    def next_batch(self):
        if self.pos + self.batch_size > len(self.indices):
            # 新 epoch：重新打乱
            self.rng.shuffle(self.indices)
            self.pos = 0
            self.epoch += 1
        batch_idx = self.indices[self.pos: self.pos + self.batch_size]
        self.pos += self.batch_size
        return [self.data[i] for i in batch_idx]


# ──────────────────────────────────────────────
# 主函数
# ──────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--num_steps", type=int, default=3000,
                        help="训练步数。8B 建议 2000-3000，3B 调试用 200-500")
    parser.add_argument("--batch_size", type=int, default=4)
    parser.add_argument("--lr", type=float, default=1e-4,
                        help="8B 模型建议 1e-4；3B 调试可用 2e-4")
    parser.add_argument("--rank", type=int, default=32,
                        help="LoRA rank。8B 建议 32-64；3B 调试用 16")
    parser.add_argument("--warmup_steps", type=int, default=100,
                        help="学习率 warmup 步数，建议为总步数的 3-5%")
    parser.add_argument("--checkpoint_name", type=str, default="multitask_v1")
    parser.add_argument("--no_publish", action="store_true")
    # 数据量控制
    # parser.add_argument("--gsm8k_samples", type=int, default=7473,
    #                     help="GSM8K 样本数，默认全量")
    # parser.add_argument("--gsm8k_upsample", type=int, default=2,
    #                     help="GSM8K 上采样倍数，补偿数量少的问题")
    # parser.add_argument("--code_samples", type=int, default=5000)
    # parser.add_argument("--ifeval_samples", type=int, default=5000)
    parser.add_argument("--max_length", type=int, default=1024,
                        help="最大序列长度。512 会截断推理链，建议至少 1024")
    parser.add_argument("--data_path", type=str, default="training_data.jsonl",
                    help="Path to pre-processed training data jsonl")
    parser.add_argument("--log_interval", type=int, default=50)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--save_every", type=int, default=500,
                    help="Save intermediate checkpoint every N steps")
    parser.add_argument("--intermediate_ttl", type=int, default=86400,
                    help="TTL in seconds for intermediate checkpoints (default: 24h)")
    parser.add_argument("--resume_from", type=str, default=None,
                    help="Resume from saved state path (tinker://...)")
    args = parser.parse_args()

    np.random.seed(args.seed)
    print("=" * 60)
    print(f"Model:         {MODEL}")
    print(f"Steps:         {args.num_steps}")
    print(f"Batch size:    {args.batch_size}")
    print(f"LR:            {args.lr}  (warmup={args.warmup_steps} steps)")
    print(f"LoRA rank:     {args.rank}")
    print(f"Max length:    {args.max_length}")
    print(f"Checkpoint:    {args.checkpoint_name}")
    print("=" * 60)

    tokenizer = get_tokenizer(MODEL)
    renderer_name = model_info.get_recommended_renderer_name(MODEL)
    renderer = renderers.get_renderer(renderer_name, tokenizer)
    print(f"Renderer: {renderer_name}\n")

    # # ── 数据加载 ──────────────────────────────
    # gsm8k_convos  = load_gsm8k_data(args.gsm8k_samples, args.gsm8k_upsample)
    # ifeval_convos = load_ifeval_data(args.ifeval_samples)
    # code_convos   = load_code_data(args.code_samples)

    # task_id: 0=gsm8k, 1=ifeval, 2=code
    # labeled = (
    #     [(c, 0) for c in gsm8k_convos] +
    #     [(c, 1) for c in ifeval_convos] +
    #     [(c, 2) for c in code_convos]
    # )
    from collections import Counter

    print("Loading pre-processed training data...")
    with open(args.data_path) as f:
        training_data = [json.loads(line) for line in f]
    print(f"  Loaded {len(training_data)} examples")

    task_names = {0: "GSM8K", 1: "IFEval", 2: "Code"}
    dist = Counter(ex["task_id"] for ex in training_data)
    for tid, name in task_names.items():
        print(f"  {name}: {dist.get(tid, 0)}")

    labeled = [(ex["messages"], ex["task_id"]) for ex in training_data]

    # ── 转换成训练格式 ─────────────────────────
    print("\nPreparing training data...")
    data_with_ids = []
    skip_counts = {"too_long": 0, "error": 0}

    for convo, task_id in labeled:
        try:
            datum = conversation_to_datum(
                convo,
                renderer,
                max_length=args.max_length,
                train_on_what=renderers.TrainOnWhat.ALL_ASSISTANT_MESSAGES
            )
            data_with_ids.append((datum, task_id))
        except Exception as e:
            err_msg = str(e).lower()
            if "length" in err_msg or "token" in err_msg:
                skip_counts["too_long"] += 1
            else:
                skip_counts["error"] += 1

    # task_names = {0: "GSM8K", 1: "IFEval", 2: "Code"}
    task_counts = {name: 0 for name in task_names.values()}
    for _, tid in data_with_ids:
        task_counts[task_names[tid]] += 1

    print(f"  Ready: {len(data_with_ids)} examples")
    print(f"  Skipped (too long): {skip_counts['too_long']} | Skipped (error): {skip_counts['error']}")
    print(f"  Per-task breakdown: {task_counts}")

    if len(data_with_ids) == 0:
        raise RuntimeError("No training data after filtering! Check data loading.")

    # ── 训练 ──────────────────────────────────
    print(f"\nCreating LoRA training client (rank={args.rank})...")
    sc = tinker.ServiceClient()
    # tc = sc.create_lora_training_client(base_model=MODEL, rank=args.rank)
    if args.resume_from:
        print(f"Resuming from: {args.resume_from}")
        tc = sc.create_training_client_from_state_with_optimizer(args.resume_from)
    else:
        tc = sc.create_lora_training_client(base_model=MODEL, rank=args.rank)
    print("  Training client ready\n")

    loader = ShuffledDataLoader(data_with_ids, args.batch_size, seed=args.seed)

    # 分任务 loss 追踪
    task_loss_accum  = {0: 0.0, 1: 0.0, 2: 0.0}
    task_loss_counts = {0: 0,   1: 0,   2: 0}

    print(f"Training for {args.num_steps} steps...")
    intermediate_checkpoints = []

    for step in range(args.num_steps):

        # 动态学习率
        lr = get_lr_with_warmup(step, args.lr, args.warmup_steps, args.num_steps)
        adam_params = types.AdamParams(
            learning_rate=lr,
            beta1=0.9,
            beta2=0.95,
            eps=1e-8
        )

        batch_with_ids = loader.next_batch()
        batch   = [item[0] for item in batch_with_ids]
        task_ids = [item[1] for item in batch_with_ids]

        fwd_bwd_future = tc.forward_backward(batch, loss_fn="cross_entropy")
        optim_future   = tc.optim_step(adam_params)

        fwd_bwd_result = fwd_bwd_future.result()
        optim_future.result()

        # 计算整体 loss
        logprobs = np.concatenate([o["logprobs"].tolist() for o in fwd_bwd_result.loss_fn_outputs])
        weights  = np.concatenate([d.loss_fn_inputs["weights"].tolist() for d in batch])
        loss = -np.dot(logprobs, weights) / max(weights.sum(), 1)

        # 按任务累积 loss（简化：整个 batch 用同一个 loss）
        # 如果 batch 里只有单一任务（理想情况），这是准确的
        for tid in task_ids:
            task_loss_accum[tid]  += loss / len(task_ids)
            task_loss_counts[tid] += 1

        if (step + 1) % args.log_interval == 0 or step == 0:
            # 计算各任务平均 loss
            task_avg = {}
            for tid in range(3):
                if task_loss_counts[tid] > 0:
                    task_avg[task_names[tid]] = task_loss_accum[tid] / task_loss_counts[tid]
                    task_loss_accum[tid]  = 0.0
                    task_loss_counts[tid] = 0

            loss_str = " | ".join([f"{k}: {v:.3f}" for k, v in task_avg.items()])
            print(
                f"  Step {step+1:4d}/{args.num_steps}"
                f" | LR: {lr:.2e}"
                f" | Loss: {loss:.4f}"
                f" | [{loss_str}]"
                f" | Epoch: {loader.epoch}"
            )

        if (step + 1) % args.save_every == 0 and (step + 1) < args.num_steps:
            ckpt_name = f"{args.checkpoint_name}_step{step+1}"
            print(f"\n  Saving intermediate checkpoint: {ckpt_name}")
            ckpt_mid = tc.save_weights_for_sampler(
                ckpt_name, ttl_seconds=args.intermediate_ttl
            ).result()
            print(f"  Saved: {ckpt_mid.path}")
            intermediate_checkpoints.append({
                "step": step + 1,
                "path": ckpt_mid.path,
                "name": ckpt_name
            })

    # ── 保存 & 发布 ───────────────────────────
    print(f"\nSaving checkpoint '{args.checkpoint_name}'...")
    ckpt = tc.save_weights_for_sampler(name=args.checkpoint_name).result()
    checkpoint_path = ckpt.path
    print(f"  Checkpoint saved: {checkpoint_path}")

    print("Saving training state for potential resume...")
    state = tc.save_state(f"{args.checkpoint_name}_state").result()
    state_path = state.path
    print(f"  State saved: {state_path}")

    if not args.no_publish:
        print("Publishing checkpoint...")
        rest_client = sc.create_rest_client()
        rest_client.publish_checkpoint_from_tinker_path(checkpoint_path).result()
        print("  Published!")

    # ── 保存实验信息 ──────────────────────────
    # info = {
    #     "checkpoint_path": checkpoint_path,
    #     "base_model": MODEL,
    #     "training": {
    #         "num_steps": args.num_steps,
    #         "batch_size": args.batch_size,
    #         "learning_rate": args.lr,
    #         "warmup_steps": args.warmup_steps,
    #         "lora_rank": args.rank,
    #         "max_length": args.max_length,
    #         "gsm8k_samples": args.gsm8k_samples,
    #         "gsm8k_upsample": args.gsm8k_upsample,
    #         "code_samples": args.code_samples,
    #         "ifeval_samples": args.ifeval_samples,
    #         "total_training_examples": len(data_with_ids),
    #         "per_task_examples": task_counts,
    #     },
    # }
    # info = {
    #     "checkpoint_path": checkpoint_path,
    #     "base_model": MODEL,
    #     "training": {
    #         "num_steps":      args.num_steps,
    #         "batch_size":     args.batch_size,
    #         "learning_rate":  args.lr,
    #         "warmup_steps":   args.warmup_steps,
    #         "lora_rank":      args.rank,
    #         "max_length":     args.max_length,
    #         "data_path":      args.data_path,
    #         "total_examples": len(data_with_ids),
    #         "per_task":       task_counts,
    #     },
    # }
    info = {
        "checkpoint_path": checkpoint_path,
        "state_path": state_path,
        "intermediate_checkpoints": intermediate_checkpoints,
        "base_model": MODEL,
        "training": {
            "num_steps":      args.num_steps,
            "batch_size":     args.batch_size,
            "learning_rate":  args.lr,
            "warmup_steps":   args.warmup_steps,
            "lora_rank":      args.rank,
            "max_length":     args.max_length,
            "data_path":      args.data_path,
            "total_examples": len(data_with_ids),
            "per_task":       task_counts,
            "resumed_from":   args.resume_from,
        },
    }
    info_path = os.path.join(EVAL_DIR, "checkpoint_info.json")
    with open(info_path, "w") as f:
        json.dump(info, f, indent=2)
    print(f"  Experiment info saved to {info_path}")

    print(f"\nDone! Evaluate with:")
    print(f'  python evaluation/eval_all.py --checkpoint_path "{checkpoint_path}" --base_model {MODEL} --limit 20')


if __name__ == "__main__":
    main()

Overwriting evaluation/train_and_publish.py


In [ ]:
# !python evaluation/train_and_publish.py \
#     --num_steps 3000 \
#     --gsm8k_samples 7473 \
#     --gsm8k_upsample 2 \
#     --code_samples 5000 \
#     --ifeval_samples 5000 \
#     --lr 1e-4 \
#     --warmup_steps 100 \
#     --rank 32 \
#     --max_length 1024 \
#     --checkpoint_name "final_8B_v3"

In [38]:
!python evaluation/train_and_publish.py \
    --data_path "training_data.jsonl" \
    --num_steps 300 \
    --batch_size 4 \
    --lr 2e-4 \
    --rank 16 \
    --warmup_steps 15 \
    --max_length 2048 \
    --checkpoint_name "debug_3b" \
    --log_interval 50 \
    --save_every 150

Model:         meta-llama/Llama-3.2-3B
Steps:         300
Batch size:    4
LR:            0.0002  (warmup=15 steps)
LoRA rank:     16
Max length:    2048
Checkpoint:    debug_3b
Renderer: role_colon

Loading pre-processed training data...
  Loaded 29999 examples
  GSM8K: 10000
  IFEval: 10000
  Code: 9999

Preparing training data...
  Ready: 29999 examples
  Skipped (too long): 0 | Skipped (error): 0
  Per-task breakdown: {'GSM8K': 10000, 'IFEval': 10000, 'Code': 9999}

Creating LoRA training client (rank=16)...
  Training client ready

Training for 300 steps...
  Step    1/300 | LR: 1.33e-05 | Loss: 1.1123 | [GSM8K: 0.278 | IFEval: 0.278 | Code: 0.278] | Epoch: 0
  Step   50/300 | LR: 1.94e-04 | Loss: 0.3794 | [GSM8K: 0.186 | IFEval: 0.235 | Code: 0.173] | Epoch: 0
  Step  100/300 | LR: 1.64e-04 | Loss: 0.9835 | [GSM8K: 0.169 | IFEval: 0.226 | Code: 0.159] | Epoch: 0
  Step  150/300 | LR: 1.18e-04 | Loss: 0.7098 | [GSM8K: 0.140 | IFEval: 0.222 | Code: 0.158] | Epoch: 0

  Saving inter

In [39]:
!python -m evaluation.eval_all \
    --checkpoint_path "tinker://34817da8-b863-5462-983d-b61fdd58423b:train:0/sampler_weights/debug_3b" \
    --base_model meta-llama/Llama-3.2-3B

INFO:__main__:Evaluating: tinker://34817da8-b863-5462-983d-b61fdd58423b:train:0/sampler_weights/debug_3b
INFO:__main__:Renderer: role_colon | temperature=0.0 top_p=1.0
INFO:__main__:============================================================
INFO:__main__:TASK 1/3: IFEval (Instruction Following)
INFO:__main__:============================================================
INFO:evaluation.eval_ifeval:Model: meta-llama/Llama-3.2-3B | Renderer: role_colon
INFO:tinker.lib.public_interfaces.service_client:ServiceClient initialized for session 187b3b69-caf8-589f-9073-d41eef9aa54b
---------------------------------------------------------                       
ifeval (541 samples): tinker-sampling/meta-llama/Llama-3.2-3B                   
max_connections: 512, fail_on_error: False, retry_on_error: 5, dataset:         
google/IFEval                                                                   
---------------------------------------------------------                       
                

In [41]:
!python evaluation/train_and_publish.py \
    --data_path "training_data.jsonl" \
    --num_steps 5000 \
    --batch_size 4 \
    --lr 1e-4 \
    --rank 32 \
    --warmup_steps 150 \
    --max_length 2048 \
    --checkpoint_name "final_8b_v1"

Model:         meta-llama/Llama-3.1-8B
Steps:         5000
Batch size:    4
LR:            0.0001  (warmup=150 steps)
LoRA rank:     32
Max length:    2048
Checkpoint:    final_8b_v1
Renderer: role_colon

Loading pre-processed training data...
  Loaded 29999 examples
  GSM8K: 10000
  IFEval: 10000
  Code: 9999

Preparing training data...
  Ready: 29999 examples
  Skipped (too long): 0 | Skipped (error): 0
  Per-task breakdown: {'GSM8K': 10000, 'IFEval': 10000, 'Code': 9999}

Creating LoRA training client (rank=32)...
  Training client ready

Training for 5000 steps...
  Step    1/5000 | LR: 6.67e-07 | Loss: 1.0114 | [GSM8K: 0.253 | IFEval: 0.253 | Code: 0.253] | Epoch: 0
  Step   50/5000 | LR: 3.33e-05 | Loss: 0.4017 | [GSM8K: 0.191 | IFEval: 0.220 | Code: 0.171] | Epoch: 0
  Step  100/5000 | LR: 6.67e-05 | Loss: 0.8565 | [GSM8K: 0.152 | IFEval: 0.197 | Code: 0.141] | Epoch: 0
  Step  150/5000 | LR: 1.00e-04 | Loss: 0.6195 | [GSM8K: 0.122 | IFEval: 0.192 | Code: 0.138] | Epoch: 0
  Ste

In [42]:
import json

with open("evaluation/checkpoint_info.json") as f:
    info = json.load(f)

ckpt = info["checkpoint_path"]
print(f"checkpoint: {ckpt}")

checkpoint: tinker://fd14953b-eff7-55e9-9c57-3154fc935ba9:train:0/sampler_weights/final_8b_v1


In [43]:
!python -m evaluation.eval_all \
    --checkpoint_path "tinker://fd14953b-eff7-55e9-9c57-3154fc935ba9:train:0/sampler_weights/final_8b_v1" \
    --base_model meta-llama/Llama-3.1-8B

INFO:__main__:Evaluating: tinker://fd14953b-eff7-55e9-9c57-3154fc935ba9:train:0/sampler_weights/final_8b_v1
INFO:__main__:Renderer: role_colon | temperature=0.0 top_p=1.0
INFO:__main__:============================================================
INFO:__main__:TASK 1/3: IFEval (Instruction Following)
INFO:__main__:============================================================
INFO:evaluation.eval_ifeval:Model: meta-llama/Llama-3.1-8B | Renderer: role_colon
INFO:tinker.lib.public_interfaces.service_client:ServiceClient initialized for session a7d88f88-1075-520c-ba8f-d3a75ea54b8c
---------------------------------------------------------                       
ifeval (541 samples): tinker-sampling/meta-llama/Llama-3.1-8B                   
max_connections: 512, fail_on_error: False, retry_on_error: 5, dataset:         
google/IFEval                                                                   
---------------------------------------------------------                       
             

In [46]:
!python -m evaluation.eval_all \
    --checkpoint_path "tinker://fd14953b-eff7-55e9-9c57-3154fc935ba9:train:0/sampler_weights/final_8b_v1_step2500" \
    --base_model meta-llama/Llama-3.1-8B

!python -m evaluation.eval_all \
    --checkpoint_path "tinker://fd14953b-eff7-55e9-9c57-3154fc935ba9:train:0/sampler_weights/final_8b_v1_step3000" \
    --base_model meta-llama/Llama-3.1-8B

!python -m evaluation.eval_all \
    --checkpoint_path "tinker://fd14953b-eff7-55e9-9c57-3154fc935ba9:train:0/sampler_weights/final_8b_v1_step3500" \
    --base_model meta-llama/Llama-3.1-8B

!python -m evaluation.eval_all \
    --checkpoint_path "tinker://fd14953b-eff7-55e9-9c57-3154fc935ba9:train:0/sampler_weights/final_8b_v1_step4000" \
    --base_model meta-llama/Llama-3.1-8B

!python -m evaluation.eval_all \
    --checkpoint_path "tinker://fd14953b-eff7-55e9-9c57-3154fc935ba9:train:0/sampler_weights/final_8b_v1_step4500" \
    --base_model meta-llama/Llama-3.1-8B

INFO:__main__:Evaluating: tinker://fd14953b-eff7-55e9-9c57-3154fc935ba9:train:0/sampler_weights/final_8b_v1_step2500
INFO:__main__:Renderer: role_colon | temperature=0.0 top_p=1.0
INFO:__main__:============================================================
INFO:__main__:TASK 1/3: IFEval (Instruction Following)
INFO:__main__:============================================================
INFO:evaluation.eval_ifeval:Model: meta-llama/Llama-3.1-8B | Renderer: role_colon
INFO:tinker.lib.public_interfaces.service_client:ServiceClient initialized for session b83b949e-23d2-581e-b71e-838f4654db91
---------------------------------------------------------                       
ifeval (541 samples): tinker-sampling/meta-llama/Llama-3.1-8B                   
max_connections: 512, fail_on_error: False, retry_on_error: 5, dataset:         
google/IFEval                                                                   
---------------------------------------------------------                       
    